# Deploy Fine-Tuned Model and Integrate with Data Agent

1. Merge LoRA adapter with base model
2. Deploy via llm-d `LLMInferenceService` on a MIG 3g.71gb slice
3. Smoke test the endpoint
4. Wire into the data-agent-template

## Setup

In [ ]:
import glob
import os

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
PVC_MOUNT_PATH = "/opt/app-root/src/shared"
MODEL_PATH = "Qwen/Qwen3-8B"
GRPO_CKPT_DIR = f"{PVC_MOUNT_PATH}/text2sql/grpo_output"
MERGED_MODEL_PATH = f"{PVC_MOUNT_PATH}/text2sql/merged_model"
NAMESPACE = "<YOUR_NAMESPACE>"

## 1. Merge LoRA Adapter

Load the GRPO-trained LoRA adapter, merge it into the base model weights,
and save the merged model for serving.

In [ ]:
checkpoint_dirs = sorted(
    glob.glob(os.path.join(GRPO_CKPT_DIR, "checkpoint-*")), key=os.path.getctime
)
CKPT_PATH = checkpoint_dirs[-1] if checkpoint_dirs else GRPO_CKPT_DIR
print(f"Loading checkpoint: {CKPT_PATH}")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="auto",
)
model = PeftModel.from_pretrained(base_model, CKPT_PATH)
model = model.merge_and_unload()
print("LoRA adapter merged")

os.makedirs(MERGED_MODEL_PATH, exist_ok=True)
model.save_pretrained(MERGED_MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.save_pretrained(MERGED_MODEL_PATH)

# Ensure group-readable
!chmod -R g+r {MERGED_MODEL_PATH}

print(f"Merged model saved to: {MERGED_MODEL_PATH}")
!du -sh {MERGED_MODEL_PATH}

## 2. Deploy with llm-d LLMInferenceService

Deploy the fine-tuned model using KServe `LLMInferenceService` with llm-d routing.
The model is served from the PVC via vLLM on a MIG 3g.71gb slice.

In [ ]:
llmisvc_manifest = f"""\
apiVersion: serving.kserve.io/v1alpha1
kind: LLMInferenceService
metadata:
  name: qwen3-8b-text2sql
  namespace: {NAMESPACE}
  annotations:
    opendatahub.io/model-type: generative
    openshift.io/display-name: Qwen3-8B Text-to-SQL (GRPO fine-tuned)
    security.opendatahub.io/enable-auth: 'false'
  labels:
    kueue.x-k8s.io/queue-name: reserved
    opendatahub.io/dashboard: 'true'
    opendatahub.io/genai-asset: 'true'
    app.kubernetes.io/part-of: llm-d
spec:
  model:
    uri: 'pvc://{PVC_MOUNT_PATH.split("/")[-1]}/text2sql/merged_model'
    name: qwen3-8b-text2sql
  replicas: 1
  router:
    scheduler: {{}}
    route: {{}}
    gateway: {{}}
  template:
    containers:
      - name: main
        env:
          - name: VLLM_ADDITIONAL_ARGS
            value: >-
              --dtype bfloat16
              --max-model-len 8192
              --max-num-seqs 8
              --gpu-memory-utilization 0.90
              --enable-auto-tool-choice
              --tool-call-parser hermes
              --enforce-eager
        resources:
          requests:
            cpu: '4'
            memory: 32Gi
            nvidia.com/mig-3g.71gb: '1'
          limits:
            cpu: '8'
            memory: 64Gi
            nvidia.com/mig-3g.71gb: '1'
"""

manifest_path = "/tmp/llminferenceservice.yaml"
with open(manifest_path, "w") as f:
    f.write(llmisvc_manifest)

print("LLMInferenceService manifest written")
print(f"Namespace: {NAMESPACE}")
print(f"Model URI: pvc://{PVC_MOUNT_PATH.split('/')[-1]}/text2sql/merged_model")

In [ ]:
!oc apply -f /tmp/llminferenceservice.yaml

In [ ]:
import time

print("Waiting for LLMInferenceService to be ready...")
for i in range(30):
    result = os.popen(
        f"oc get llminferenceservice qwen3-8b-text2sql -n {NAMESPACE} "
        "-o jsonpath='{.status.conditions[?(@.type==\"Ready\")].status}'"
    ).read().strip()
    print(f"[{i * 20}s] Ready: {result}")
    if result == "True":
        break
    time.sleep(20)

# Get endpoint URL
endpoint = os.popen(
    f"oc get llminferenceservice qwen3-8b-text2sql -n {NAMESPACE} "
    "-o jsonpath='{.status.url}'"
).read().strip()
print(f"\nEndpoint: {endpoint}")

## 3. Smoke Test

In [ ]:
from openai import OpenAI

MODEL_ENDPOINT = endpoint or "http://localhost:8000"  # fallback for local testing

client = OpenAI(
    base_url=f"{MODEL_ENDPOINT}/v1",
    api_key="not-required",
)

test_questions = [
    "How many influenza notifications were there in NSW in 2023?",
    "Which state had the highest salmonellosis rate per 100,000 in 2024?",
    "Show influenza trends from 2020 to 2025 across all states.",
    "Compare all disease notifications in Victoria for the most recent year.",
    "What is the per-capita meningococcal disease rate in Queensland for 2023?",
]

from reward.nndss_schema import NNDSS_DDL

print("=== Smoke Test Results ===")
for i, question in enumerate(test_questions, 1):
    response = client.chat.completions.create(
        model="qwen3-8b-text2sql",
        messages=[
            {"role": "system", "content": "You are a SQL expert. Write a SQL query that answers the question. Output only the SQL."},
            {"role": "user", "content": f"Schema:\n{NNDSS_DDL}\n\nQuestion: {question}"},
        ],
        temperature=0.1,
        max_tokens=256,
    )
    sql = response.choices[0].message.content
    print(f"\n[{i}] {question}")
    print(f"    SQL: {sql[:200]}")

## 4. Wire into Data Agent Template

The data-agent-template uses `ChatOpenAI` configured via environment variables.
Update the model name and endpoint to point to the fine-tuned model.

In [ ]:
print("=== Data Agent Integration ===")
print()
print("Option A: Environment variables")
print(f'  export MODEL_NAME="qwen3-8b-text2sql"')
print(f'  export MODEL_ENDPOINT="{MODEL_ENDPOINT}/v1"')
print()
print("Option B: set-model.sh (on OpenShift)")
print(f'  ./scripts/set-model.sh qwen3-8b-text2sql {MODEL_ENDPOINT}/v1')
print()
print("Option C: agent-config.yaml")
print("  deployment:")
print("    model_name: qwen3-8b-text2sql")
print(f"    model_endpoint: {MODEL_ENDPOINT}/v1")

### End-to-End Integration Test

If the data agent is running, send a test question through the full pipeline:
user question -> fine-tuned model generates SQL -> Trino executes -> agent responds.

In [ ]:
# Uncomment and set the agent URL to test end-to-end:

# AGENT_URL = "https://<agent-route>/api/chat"
# import requests
# response = requests.post(AGENT_URL, json={
#     "message": "What was the influenza notification rate per 100,000 in NSW for 2023?"
# })
# print(response.json())

## Next Steps

- **Graduate to Qwen3-27B**: Update `MODEL_PATH`, increase MIG slices to 2-3
- **Multi-turn**: Add ReViSQL's multi-turn interaction (up to 5 turns model <-> database)
- **VeriEQL**: Add formal SQL equivalence checking as additional reward signal
- **MaaS governance**: Apply `maas-modelref.yaml` and `maas-subscription.yaml` for rate-limited API access
- **GuideLLM benchmark**: Run EvalHub `guidellm` to measure serving performance